In [4]:
import pandas as pd
import numpy as np

from linearmodels import PanelOLS

In [2]:
data = pd.read_csv('cleaned_school_data.csv')  

In [5]:
print(data.groupby("Year")["mean_scale_score"].agg(["mean", "std", "min", "max"]).round(3))

         mean    std    min    max
Year                              
2018  600.497  9.411  574.0  632.0
2019  599.576  9.569  559.0  634.0
2022  599.954  9.668  578.0  631.0


In [ ]:
# adjust % poverty to a percentage scale
data['poverty_percentage'] = (data['% Poverty'] * 100).round(3)

In [ ]:
# preliminary DiD variables
data['post'] = (data['Year'] == 2022).astype(int)
data['treated_cont'] = data['post'] * data['poverty_percentage']

In [ ]:
# Time-placebo test: use 2018 and 2019 as "pre" period
# this is a method to test the parallel trends assumption by checking for any pre-existing trends in the outcome variable before the treatment period
pre_data = data[data['Year'].isin([2018,2019])].copy() # new pre data
pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage'] # new placebo interaction term for treated

In [ ]:
pre_data = pre_data.set_index(['DBN', 'Year']) # set index for PanelOLS, it requires a multi-index with entity and time dimensions

In [ ]:
# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model = PanelOLS(
    dependent=pre_data['mean_scale_score'],
    exog=pre_data[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [16]:
print(f"Placebo test coefficient: {parallel_trends_model.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: -0.0009, p-value: 0.8371


```python
Placebo test coefficient: -0.0009, p-value: 0.8371
``` 

This is great!!!! Near zero trend in pre-period, parallel trends assumption met